# ChromaDB Setup + Document Ingestion

## Problem Statement
Build a semantic knowledge base of Python best practices that retrieves the right rule when queried with a bug description.

## My Approach
Store 10 Python best-practice rules (covering mutable defaults, file handling, imports, function design, etc.) in a local ChromaDB persistent collection. Use semantic search to verify the right rule surfaces for each bug type found in the sample file.

## What I Learned
ChromaDB finds documents by meaning, not keywords. Distance score tells you retrieval confidence — lower is better. get_or_create_collection + a count-check is the production pattern for idempotent pipelines.

## Where I Got Stuck
The spot-check used "id_0" instead of "rule_001" - returned empty silently, no error. That's the dangerous kind of bug. Always verify IDs match exactly what you ingested.

## What I'd Do Differently
Test all 4 bug queries from the start, not just the easy ones. And I'd print the distance threshold clearly — a distance of 1.5+ means the model is guessing, not matching.

In [ ]:
import chromadb

In [2]:
RULES = [
    {
        "id": "rule_001",
        "text": "Never use mutable objects (lists, dicts) as default argument values. They persist across calls. Use None and initialize inside the function body.",
        "metadata": {"source": "pep8", "category": "defaults"}
    },
    {
        "id": "rule_002", 
        "text": "Always use a context manager (with statement) when opening files. It guarantees the file is closed even if an exception is raised.",
        "metadata": {"source": "clean-code", "category": "resources"}
    },
    {
        "id": "rule_003",
        "text": "Avoid imports inside functions unless necessary to prevent circular dependencies. Standard practice is to keep all imports at the top of the file.",
        "metadata": {"source": "pep8", "category": "imports"}
    },
    {
        "id": "rule_004",
        "text": "Keep function parameters to a minimum (ideally 3 or fewer). If you need more, consider passing a single object or dictionary to group related data.",
        "metadata": {"source": "clean-code", "category": "design"}
    },
    {
        "id": "rule_005",
        "text": "Never use a bare 'except:'. Always catch specific exceptions (e.g., ValueError, KeyError) to avoid accidentally swallowing keyboard interrupts or system exits.",
        "metadata": {"source": "pep8", "category": "exceptions"}
    },
    {
        "id": "rule_006",
        "text": "Use type hints (PEP 484) to make code more readable and allow static analysis tools like Mypy to catch bugs before runtime.",
        "metadata": {"source": "typing", "category": "quality"}
    },
    {
        "id": "rule_007",
        "text": "Follow PEP 8 naming: snake_case for functions/variables, PascalCase for classes, and SCREAMING_SNAKE_CASE for constants.",
        "metadata": {"source": "pep8", "category": "naming"}
    },
    {
        "id": "rule_008",
        "text": "Avoid 'magic numbers'. Use named constants to explain the significance of a numeric value instead of hardcoding it in logic.",
        "metadata": {"source": "clean-code", "category": "readability"}
    },
    {
        "id": "rule_009",
        "text": "Minimize the use of 'global' state. Global variables make code harder to test and debug; prefer passing parameters or using class attributes.",
        "metadata": {"source": "clean-code", "category": "design"}
    },
    {
        "id": "rule_010",
        "text": "Write docstrings (PEP 257) for all public modules, functions, and classes. Use triple double-quotes and follow a consistent style like Google or NumPy.",
        "metadata": {"source": "pep257", "category": "documentation"}
    }
]

In [17]:
# 1. Initialize PersistentClient (No Docker needed) which saves the data to the './chroma_db' folder in your environment
client = chromadb.PersistentClient(path="./chroma_db")

# 2. Create or Get Collection
# Using get_or_create prevents errors when re-running the notebook cell
collection = client.get_or_create_collection(name="python_best_practices")

#My solution
# # Prepare the parallel lists
# documents = [r["text"] for r in RULES]
# ids = [r["id"] for r in RULES]
# metadatas = [r["metadata"] for r in RULES]

# # Add to the local persistent store
# collection.add(
#     documents=documents,
#     ids=ids,
#     metadatas=metadatas
# )


# --- Idempotent ingestion guard ---
# Refactored soultion
# If collection already has data, skip. This makes the notebook safe to re-run.
if collection.count() > 0:
    print(f"Collection already loaded ({collection.count()} docs) — skipping ingestion.")
else:
    collection.add(
        documents=[r["text"] for r in RULES],
        ids=[r["id"] for r in RULES],
        metadatas=[r["metadata"] for r in RULES]
    )
    print(f"Ingested {collection.count()} rules into ChromaDB.")


Collection already loaded (10 docs) — skipping ingestion.


In [16]:
#verification
print(f"Total documents in collection: {collection.count()}")

# Spot-check to ensure data is actually there
check_result = collection.get(ids=["rule_001"])
print(f"Spot-check (id_0): {check_result['documents']}")

Total documents in collection: 10
Spot-check (id_0): ['Never use mutable objects (lists, dicts) as default argument values. They persist across calls. Use None and initialize inside the function body.']


In [18]:
# 6. Querying
QUERIES = [
    "mutable default argument in function signature",          # bug: results=[]
    "file opened without context manager",                     # bug: open() without with
    "import statement inside function body",                   # bug: import requests inside fn
    "function has too many parameters",                        # bug: 5 params in fetch_data
]
for q in QUERIES :
    query_results = collection.query(
        query_texts=[q],
        n_results=3
    )

    print(f"\nQuery: {q}")
    print(f"Top 3 Results:")
    for i in range(len(query_results['documents'][0])):
        print(f"[{i}] (distance: {query_results['distances'][0][i]}) [{query_results['metadatas'][0][i]['source']}/{query_results['metadatas'][0][i]['category']}] {query_results['documents'][0][i]}")
        


Query: mutable default argument in function signature
Top 3 Results:
[0] (distance: 0.819106936454773) [pep8/defaults] Never use mutable objects (lists, dicts) as default argument values. They persist across calls. Use None and initialize inside the function body.
[1] (distance: 1.406186580657959) [clean-code/design] Keep function parameters to a minimum (ideally 3 or fewer). If you need more, consider passing a single object or dictionary to group related data.
[2] (distance: 1.4113056659698486) [pep8/naming] Follow PEP 8 naming: snake_case for functions/variables, PascalCase for classes, and SCREAMING_SNAKE_CASE for constants.

Query: file opened without context manager
Top 3 Results:
[0] (distance: 0.6286314129829407) [clean-code/resources] Always use a context manager (with statement) when opening files. It guarantees the file is closed even if an exception is raised.
[1] (distance: 1.5371534824371338) [pep8/imports] Avoid imports inside functions unless necessary to prevent circu